# 10 · Final report: test scores, significance, breakdowns, errors

**Goal:** score the chosen configurations on **test** (saved blind since Direction 1), compare them with paired tests, and produce the tables of the written report. No configuration is chosen here: every choice was made on val (B, D1, D2) or on train CV + val (D3). A new run (e.g. a later cross-encoder) only needs one line in `RUNS`.

**Sections:** 1 data · 2 main table (val and test, every run) · 3 headline table (mean over seeds) · 4 paired tests · 5 by query type · 6 where each label lands · 7 per-query wins and losses · 8 worst test queries of the final model · 9 save.

**Kaggle settings:** CPU, no internet, a few minutes. Also runs locally from `notebooks/` (inputs read from `../../kaggle_outputs`).

**Inputs** (Add Input):
- Dataset `wm-src`, a version whose `data/results/` has the `d3_*` runs. Required.
- Notebook Output **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.
- Notebook Output **07_oof** (assemble version) or the dataset with `ce_deberta_main_ce4.parquet`: test scores of the 5-fold CE. Required.

**Output** (`/kaggle/working/report/`, locally `data/report/`): `results_all.csv`, `results_headline.csv`, `paired_tests.csv`, `by_source.csv`, `label_ranks.csv`, `query_deltas.csv`, `worst_queries.csv`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.data import PREPROCESS_HINT, find
from src.evaluation import paired_bootstrap, summarize

ON_KAGGLE = Path('/kaggle/input').exists()
INPUT = '/kaggle/input' if ON_KAGGLE else '../../kaggle_outputs'
OUT = Path('/kaggle/working/report') if ON_KAGGLE else ROOT / 'data' / 'report'
RES = ROOT / 'data' / 'results'
CE5 = 'd3_ce_deberta_main_ce4'          # 5-fold CE: val from its run file, test from the assembled 07 file
FINAL = 'd3_lgbm_multiclass'            # final system (chosen on CV + val in 09)

# (stage, configuration, seed, run name); order = order of the tables
RUNS = [
    ('B0', 'Random order', None, 'b0_random'),
    ('B1', 'BM25 title', None, 'b1_bm25_title'),
    ('B1', 'BM25 main', None, 'b1_bm25_main'),
    ('B2', 'Bi-encoder MiniLM-L6 title (zero-shot)', None, 'b2_minilml6_title'),
    ('B2', 'Bi-encoder MiniLM-L6 main (zero-shot)', None, 'b2_minilml6_main'),
    ('B2', 'Bi-encoder bge-base title (zero-shot)', None, 'b2_bge_title'),
    ('B2', 'Bi-encoder bge-base main (zero-shot)', None, 'b2_bge_main'),
    ('B3', 'CE MiniLM MS MARCO title (zero-shot)', None, 'b3_minilm_title_zeroshot'),
    ('B3', 'CE MiniLM title, MSE', 42, 'b3_minilm_title_mse_s42'),
    ('B3', 'CE MiniLM title, MSE', 43, 'b3_minilm_title_mse_s43'),
    ('D1', 'CE MiniLM title_bc, MSE', 42, 'd1_minilm_title_bc_mse_s42'),
    ('D1', 'CE MiniLM title_bc, MSE', 43, 'd1_minilm_title_bc_mse_s43'),
    ('D1', 'CE MiniLM main, MSE', 42, 'd1_minilm_main_mse_s42'),
    ('D1', 'CE MiniLM main, MSE', 43, 'd1_minilm_main_mse_s43'),
    ('D1', 'CE DeBERTa title, MSE', 42, 'd1_deberta_title_mse_s42'),
    ('D1', 'CE DeBERTa main, MSE', 42, 'd1_deberta_main_mse_s42'),
    ('D1', 'CE DeBERTa main, MSE', 43, 'd1_deberta_main_mse_s43'),
    ('D2', 'CE MiniLM title, ce4', 42, 'd2_minilm_title_ce4_s42'),
    ('D2', 'CE MiniLM title, ce4', 43, 'd2_minilm_title_ce4_s43'),
    ('D2', 'CE MiniLM title, ListNet', 42, 'd2_minilm_title_listnet_s42'),
    ('D2', 'CE MiniLM title, ListNet', 43, 'd2_minilm_title_listnet_s43'),
    ('D2', 'CE DeBERTa main, ce4', 42, 'd2_deberta_main_ce4_s42'),
    ('D3', 'CE DeBERTa main ce4, mean of 5 fold models', 42, CE5),
    ('D3', 'LightGBM lambdarank on CE + features', 42, 'd3_lgbm_lambdarank'),
    ('D3', 'LightGBM multiclass on CE + features', 42, 'd3_lgbm_multiclass'),
]

# (question, run a, run b): paired difference b - a on the same queries
PAIRS = [
    ('B2 bge main vs BM25 main', 'b1_bm25_main', 'b2_bge_main'),
    ('B3 fine-tuning vs zero-shot CE', 'b3_minilm_title_zeroshot', 'b3_minilm_title_mse_s42'),
    ('B3 vs BM25 main', 'b1_bm25_main', 'b3_minilm_title_mse_s42'),
    ('D1 MiniLM: title_bc vs title (s42)', 'b3_minilm_title_mse_s42', 'd1_minilm_title_bc_mse_s42'),
    ('D1 MiniLM: main vs title (s42)', 'b3_minilm_title_mse_s42', 'd1_minilm_main_mse_s42'),
    ('D1 DeBERTa: main vs title', 'd1_deberta_title_mse_s42', 'd1_deberta_main_mse_s42'),
    ('D1 main: DeBERTa vs MiniLM (s42)', 'd1_minilm_main_mse_s42', 'd1_deberta_main_mse_s42'),
    ('D1 main: DeBERTa vs MiniLM (s43)', 'd1_minilm_main_mse_s43', 'd1_deberta_main_mse_s43'),
    ('D1 seed: DeBERTa main s43 vs s42', 'd1_deberta_main_mse_s42', 'd1_deberta_main_mse_s43'),
    ('D2 MiniLM: ce4 vs MSE (s42)', 'b3_minilm_title_mse_s42', 'd2_minilm_title_ce4_s42'),
    ('D2 MiniLM: ce4 vs MSE (s43)', 'b3_minilm_title_mse_s43', 'd2_minilm_title_ce4_s43'),
    ('D2 MiniLM: ListNet vs MSE (s42)', 'b3_minilm_title_mse_s42', 'd2_minilm_title_listnet_s42'),
    ('D2 MiniLM: ListNet vs MSE (s43)', 'b3_minilm_title_mse_s43', 'd2_minilm_title_listnet_s43'),
    ('D2 DeBERTa main: ce4 vs MSE', 'd1_deberta_main_mse_s42', 'd2_deberta_main_ce4_s42'),
    ('D3 mean of 5 fold models vs 1 model on all train', 'd2_deberta_main_ce4_s42', CE5),
    ('D3 LightGBM lambdarank vs CE (5 folds)', CE5, 'd3_lgbm_lambdarank'),
    ('D3 LightGBM multiclass vs CE (5 folds)', CE5, 'd3_lgbm_multiclass'),
    ('D3 multiclass vs lambdarank', 'd3_lgbm_lambdarank', 'd3_lgbm_multiclass'),
    ('Final vs B3', 'b3_minilm_title_mse_s42', FINAL),
    ('Final vs BM25 main', 'b1_bm25_main', FINAL),
]
BREAKDOWN = ['b1_bm25_main', 'b2_bge_main', 'b3_minilm_title_mse_s42', 'd1_deberta_main_mse_s42', CE5, FINAL]
NAME = {r: (f'{c} (s{s})' if s and sum(x[1] == c for x in RUNS) > 1 else c) for _, c, s, r in RUNS}

## 1. Data and scores

Labels come from `pairs.parquet`. Every run's scores are aligned to the pairs by `example_id` and must cover every pair of the split.

In [ ]:
pairs = pd.read_parquet(find('pairs.parquet', PREPROCESS_HINT, INPUT),
                        columns=['example_id', 'query_id', 'query', 'product_id', 'esci_label', 'source', 'split'])
parts = {s: pairs[pairs['split'] == s].sort_values(['query_id', 'example_id']).reset_index(drop=True) for s in ['val', 'test']}
ce5_test = pd.read_parquet(find('ce_deberta_main_ce4.parquet', 'Add Input -> 07_oof (assemble) output', INPUT))
ce5_test = ce5_test[ce5_test['split'] == 'test'][['example_id', 'score']]


def scores(run, split):
    f = RES / 'scores' / f'{run}_{split}.parquet'
    if f.exists():
        sc = pd.read_parquet(f, columns=['example_id', 'score'])
    elif run == CE5 and split == 'test':
        sc = ce5_test
    else:
        raise FileNotFoundError(f'{f.name}: rebuild wm-src with the latest data/results')
    s = parts[split][['example_id']].merge(sc, on='example_id', how='left')['score']
    assert s.notna().all(), f'{run} {split}: scores do not cover every pair'
    return s.to_numpy()


metrics, per_query, score_cache = {}, {}, {}
for _, _, _, run in RUNS:
    for split in ['val', 'test']:
        score_cache[run, split] = scores(run, split)
        metrics[run, split], per_query[run, split] = summarize(parts[split], score_cache[run, split])
print(len(RUNS), 'runs scored on val and test |', {s: f"{p['query_id'].nunique():,} queries" for s, p in parts.items()})

## 2. Every run (val and test)

nDCG with official gains (E 1, S 0.1, C 0.01, I 0), ties broken at random (3 seeds), 95% bootstrap CI over queries. `ndcg_amazon` uses the gains of Amazon's script (S and C swapped), for comparison with the paper only.

In [ ]:
rows = []
for stage, config, seed, run in RUNS:
    v, t = metrics[run, 'val'], metrics[run, 'test']
    rows.append({'stage': stage, 'configuration': config, 'seed': seed, 'run': run,
                 'val_ndcg': v['ndcg'], 'test_ndcg': t['ndcg'], 'test_ci95': t['ndcg_ci95'],
                 'test_ndcg_amazon': t['ndcg_amazon'], 'test_ndcg10': t['ndcg10'], 'test_mrr_e': t['mrr_e'],
                 'test_e_over_s': t['e_over_s']})
results = pd.DataFrame(rows)
pd.set_option('display.width', 250, 'display.max_columns', 30, 'display.max_colwidth', 60)
display(results.drop(columns='run'))

## 3. Headline table (mean over seeds)

One row per configuration; configurations run with two seeds show the mean and the seed range on test.

In [ ]:
headline = (results.groupby(['stage', 'configuration'], sort=False)
            .agg(seeds=('seed', lambda s: ', '.join(str(int(x)) for x in s.dropna()) or '-'),
                 val_ndcg=('val_ndcg', 'mean'), test_ndcg=('test_ndcg', 'mean'),
                 test_min=('test_ndcg', 'min'), test_max=('test_ndcg', 'max'),
                 test_ndcg10=('test_ndcg10', 'mean'), test_mrr_e=('test_mrr_e', 'mean'),
                 test_e_over_s=('test_e_over_s', 'mean'))
            .round(4).reset_index())
display(headline)

## 4. Paired tests

Difference b − a of per-query nDCG on the same queries: mean, 95% paired bootstrap CI, and the Wilcoxon signed-rank p-value (queries with equal nDCG dropped). Val is shown next to test to check that what was seen on val holds on test.

In [ ]:
rows = []
for question, a, b in PAIRS:
    r = {'question': question, 'a': NAME[a], 'b': NAME[b]}
    for split in ['val', 'test']:
        pa, pb = per_query[a, split], per_query[b, split]
        p = paired_bootstrap(pa, pb)
        d = (pb.set_index('query_id')['ndcg'] - pa.set_index('query_id')['ndcg']).dropna()
        r[f'{split}_diff'], r[f'{split}_ci95'] = p['diff'], p['ci95']
        if split == 'test':
            r['test_wilcoxon_p'] = float(wilcoxon(d[d != 0]).pvalue) if (d != 0).sum() > 10 else np.nan
            r['test_better'], r['test_worse'] = int((d > 1e-9).sum()), int((d < -1e-9).sum())
    rows.append(r)
paired = pd.DataFrame(rows)
paired['test_wilcoxon_p'] = paired['test_wilcoxon_p'].map(lambda p: f'{p:.1e}')
display(paired)

## 5. By query type

`source` from the dataset: `other` (most queries), `parse_pattern` (structured queries), `negations` ("without", "not", …), `behavioral`, `nlqec` (natural-language queries, very few). Mean test nDCG per type.

In [ ]:
src_of = parts['test'].groupby('query_id')['source'].first()
by_source = pd.DataFrame({NAME[r]: per_query[r, 'test'].set_index('query_id')['ndcg'] for r in BREAKDOWN})
by_source['source'] = src_of.reindex(by_source.index)
by_source = by_source.groupby('source').agg(['mean']).droplevel(1, axis=1).round(4)
by_source.insert(0, 'queries', src_of.value_counts())
display(by_source.sort_values('queries', ascending=False))

## 6. Where each label lands

Mean position of each label inside its list, as a share of the list (0 = top, 1 = bottom; random ≈ 0.5), on test. Exact items should rise and Irrelevant items sink; the gap between E and S is what nDCG rewards most.

In [ ]:
def label_positions(run):
    d = parts['test'][['query_id', 'esci_label']].assign(score=score_cache[run, 'test'])
    r = d.groupby('query_id')['score'].rank(ascending=False, method='average')
    n = d.groupby('query_id')['score'].transform('size')
    return ((r - 1) / (n - 1)).groupby(d['esci_label']).mean()


label_ranks = pd.DataFrame({NAME[r]: label_positions(r) for r in BREAKDOWN}).T[['E', 'S', 'C', 'I']].round(3)
label_ranks['S - E'] = (label_ranks['S'] - label_ranks['E']).round(3)
display(label_ranks)

## 7. Per-query wins and losses: final system vs B3

How many test queries the final system improves or hurts compared with the B3 reproduction, and the size of the changes.

In [ ]:
qa = per_query['b3_minilm_title_mse_s42', 'test'].set_index('query_id')['ndcg']
qb = per_query[FINAL, 'test'].set_index('query_id')['ndcg']
delta = (qb - qa).dropna().rename('delta').to_frame()
delta['b3'], delta['final'] = qa, qb
info = parts['test'].groupby('query_id').agg(query=('query', 'first'), source=('source', 'first'), n=('esci_label', 'size'),
                                              n_E=('esci_label', lambda l: int((l == 'E').sum())),
                                              n_S=('esci_label', lambda l: int((l == 'S').sum())))
delta = delta.join(info)
bins = pd.cut(delta['delta'], [-1, -0.1, -0.01, -1e-9, 1e-9, 0.01, 0.1, 1],
              labels=['< -0.1', '-0.1..-0.01', '-0.01..0', 'same', '0..0.01', '0.01..0.1', '> 0.1'])
display(bins.value_counts().reindex(bins.cat.categories).rename('queries').to_frame().T)
show = ['query', 'source', 'n', 'n_E', 'n_S', 'b3', 'final', 'delta']
print('Largest gains'); display(delta.sort_values('delta', ascending=False)[show].head(8).round(3))
print('Largest losses'); display(delta.sort_values('delta')[show].head(8).round(3))

## 8. Worst test queries of the final system

The lowest-nDCG queries that have at least one Exact item, with the full ranked list (label, score, title). Used for the error analysis of the report.

In [ ]:
titles = pd.read_parquet(find('products_clean.parquet', PREPROCESS_HINT, INPUT), columns=['product_id', 'title'])
titles = titles.set_index('product_id')['title']
t = parts['test'].assign(score=score_cache[FINAL, 'test'])
pq = per_query[FINAL, 'test'].set_index('query_id')
has_e = t.groupby('query_id')['esci_label'].apply(lambda l: (l == 'E').any())
worst = pq.loc[has_e[has_e].index, 'ndcg'].sort_values().head(6)

rows = []
for qid, nd in worst.items():
    q = t[t['query_id'] == qid].sort_values('score', ascending=False)
    print(f"\n[{q['source'].iloc[0]}] \"{q['query'].iloc[0]}\"  nDCG {nd:.3f}  ({len(q)} candidates)")
    for rank, (_, r) in enumerate(q.iterrows(), 1):
        title = str(titles.get(r['product_id'], ''))[:90]
        if rank <= 8 or r['esci_label'] == 'E':
            print(f"  {rank:2d}. {r['esci_label']}  {r['score']:.3f}  {title}")
        rows.append({'query_id': qid, 'query': r['query'], 'ndcg': round(nd, 4), 'rank': rank,
                     'label': r['esci_label'], 'score': round(float(r['score']), 4), 'title': title})
worst_rows = pd.DataFrame(rows)

## 9. Save

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT / 'results_all.csv', index=False)
headline.to_csv(OUT / 'results_headline.csv', index=False)
paired.to_csv(OUT / 'paired_tests.csv', index=False)
by_source.to_csv(OUT / 'by_source.csv')
label_ranks.to_csv(OUT / 'label_ranks.csv')
delta.sort_values('delta').to_csv(OUT / 'query_deltas.csv')
worst_rows.to_csv(OUT / 'worst_queries.csv', index=False)
print('wrote', sorted(p.name for p in OUT.glob('*.csv')), 'to', OUT)